# Module 4: Feature Engineering
**Real-Time Payment Fraud Detection for an E-Commerce Platform**

All feature engineering, encoding, temporal splitting, and imbalance-handling choices below strictly follow the EDA findings (`EDA-E-commerce_Fraud_Detection.ipynb`):
1. **Account Recurrence & Destination Indicators (`[Q1.1]`, `[Q3.2]`, `[Q3.3]`)**:
   * Senders (`nameOrig`) are single-use in `99.85%` of rows -> sender velocity features are infeasible.
   * Destination accounts (`nameDest`) recur in `97.31%` of rows, and `"Destination account never seen before"` (`dest_first_seen = prior_dest == 0`) is deployable at initiation time, yielding a **10.35x fraud rate lift** (`1.32%` vs `0.13%`) and capturing `62.99%` of all fraud cases.
2. **Transaction Amount Signals (`[Q3.1]`)**:
   * Fraud rate has a U-shaped relationship with amount; high amount bands ($\ge 1,000,000$) hold `86.76%` of total fraud value in just `4.71%` of `TRANSFER` and `CASH_OUT` transactions.
3. **Time Windows & Proposed Temporal Split (`[S0]`, `[Q1.1]`, `[Q2.3]`)**:
   * `hour = (step - 1) % 24`, `day = (step - 1) // 24 + 1`, `week = (step - 1) // 168 + 1`.
   * Time-ordered split (`SPLIT_STEPS = (504, 624)`): **Train** = steps `1–504` (days 1–21), **Validation** = steps `505–624` (days 22–26), **Test** = steps `625–743` (days 27–31).

In [1]:
# ===== [M4.0] Setup & Load Cleaned Dataset from Module 3 =====
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
np.random.seed(42)

DATA_PATH = r"C:\LONG\Project\BA Project\PS_20174392719_1491204439457_log.csv"
OUT_DIR = r"C:\LONG\Project\BA Project\outputs"
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
CLEAN_PATH = f"{OUT_DIR}/paysim_cleaned_module3.parquet"
SPLIT_STEPS = (504, 624)  # Exact match with [S0] in EDA notebook

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.4f}".format)

if os.path.exists(CLEAN_PATH):
    df = pd.read_parquet(CLEAN_PATH)
else:
    raise FileNotFoundError(f"Please run Module 3 first to generate {CLEAN_PATH}")

print(f"Loaded cleaned modelling population (TRANSFER + CASH_OUT): {len(df):,} rows")

Loaded cleaned modelling population (TRANSFER + CASH_OUT): 2,770,409 rows


In [2]:
# ===== [M4.1] Construct Deployable Initiation-Time Features (Aligned with [Q1.1], [Q3.1]-[Q3.3] & Synthetic Telemetry) =====
# Ensure stable step order as in [Q1.1]
if not df["step"].is_monotonic_increasing:
    df = df.sort_values("step", kind="stable").reset_index(drop=True)

# 1. Time window features (exact formulas from [Q1.1])
df["hour"] = ((df["step"] - 1) % 24).astype("int8")
df["day"]  = ((df["step"] - 1) // 24 + 1).astype("int8")
df["week"] = ((df["step"] - 1) // 168 + 1).astype("int8")
df["window"] = pd.Categorical(
    np.select([df["step"] <= SPLIT_STEPS[0], df["step"] <= SPLIT_STEPS[1]], ["train", "validation"], default="test"),
    categories=["train", "validation", "test"]
)

# 2. Destination recurrence features (exact logic from [Q1.1] & [Q3.2]-[Q3.3])
dest_codes, _ = pd.factorize(df["nameDest"])
prior_dest = pd.Series(dest_codes).groupby(dest_codes).cumcount().to_numpy().astype("int32")
df["prior_dest"] = prior_dest
df["dest_first_seen"] = (prior_dest == 0).astype("int8")

# Time since last transaction to the same destination account (in hours/steps)
prev_dest_step = df.groupby(dest_codes)["step"].shift(1)
df["steps_since_last_dest_tx"] = (df["step"] - prev_dest_step).fillna(-1).astype("int32")

# 3. Amount transformations & high-value band indicator (from [Q3.1])
df["log_amount"] = np.log1p(df["amount"]).astype("float32")
df["is_high_amount_ge_1M"] = (df["amount"] >= 1_000_000).astype("int8")

# 4. Categorical encoding: TRANSFER (1) vs CASH_OUT (0)
df["is_transfer"] = (df["type"] == "TRANSFER").astype("int8")

# 5. Log-transform skewed synthetic distance
df["log_ip_billing_distance_km"] = np.log1p(df["ip_billing_distance_km"]).astype("float32")

# Drop raw string account identifiers and categorical type after encoding
df.drop(columns=["nameOrig", "nameDest", "type"], inplace=True)
display(df.head())

,step,amount,isFraud,account_age_days,device_trust_score,is_new_device,ip_billing_distance_km,shipping_billing_mismatch,failed_auth_attempts_24h,is_amount_outlier_3xIQR,is_zero_amount,hour,day,week,window,prior_dest,dest_first_seen,steps_since_last_dest_tx,log_amount,is_high_amount_ge_1M,is_transfer,log_ip_billing_distance_km
0,1,181.0000,1,4,0.1252,1,232.1133,1,3,0,0,0,1,1,train,0,1,-1,5.2040,0,1,5.4515
1,1,181.0000,1,28,0.3483,1,254.3392,0,2,0,0,0,1,1,train,0,1,-1,5.2040,0,0,5.5426
2,1,"229,133.9400",0,510,0.6219,0,5.2976,0,0,0,0,0,1,1,train,0,1,-1,12.3421,0,0,1.8402
3,1,"215,310.3000",0,230,0.9281,0,19.1777,0,0,0,0,0,1,1,train,0,1,-1,12.2798,0,1,3.0046
4,1,"311,685.8900",0,654,0.7913,0,1.1046,0,0,0,0,0,1,1,train,0,1,-1,12.6498,0,1,0.7441


In [3]:
# ===== [M4.2] Time-Ordered Split (Train: 1-504, Validation: 505-624, Test: 625-743) & Numerical Scaling =====
FEATURE_COLS = [
    "amount", "log_amount", "is_high_amount_ge_1M", "is_zero_amount", "is_amount_outlier_3xIQR",
    "is_transfer", "hour", "day",
    "prior_dest", "dest_first_seen", "steps_since_last_dest_tx",
    "account_age_days", "device_trust_score", "is_new_device",
    "ip_billing_distance_km", "log_ip_billing_distance_km",
    "shipping_billing_mismatch", "failed_auth_attempts_24h"
]

train_mask = df["window"] == "train"
val_mask   = df["window"] == "validation"
test_mask  = df["window"] == "test"

X_train, y_train = df.loc[train_mask, FEATURE_COLS].copy(), df.loc[train_mask, "isFraud"].copy()
X_val,   y_val   = df.loc[val_mask,   FEATURE_COLS].copy(), df.loc[val_mask,   "isFraud"].copy()
X_test,  y_test  = df.loc[test_mask,  FEATURE_COLS].copy(), df.loc[test_mask,  "isFraud"].copy()

# Preserve unscaled transaction amounts for value-weighted KPI evaluation in Module 5
amt_train = df.loc[train_mask, "amount"].to_numpy().copy()
amt_val   = df.loc[val_mask,   "amount"].to_numpy().copy()
amt_test  = df.loc[test_mask,  "amount"].to_numpy().copy()

# Scale numerical features using RobustScaler fitted strictly on the training window
NUM_COLS_TO_SCALE = [
    "amount", "log_amount", "prior_dest", "steps_since_last_dest_tx",
    "account_age_days", "ip_billing_distance_km", "log_ip_billing_distance_km", "failed_auth_attempts_24h"
]
scaler = RobustScaler()
X_train[NUM_COLS_TO_SCALE] = scaler.fit_transform(X_train[NUM_COLS_TO_SCALE]).astype("float32")
X_val[NUM_COLS_TO_SCALE]   = scaler.transform(X_val[NUM_COLS_TO_SCALE]).astype("float32")
X_test[NUM_COLS_TO_SCALE]  = scaler.transform(X_test[NUM_COLS_TO_SCALE]).astype("float32")

split_table = pd.DataFrame([
    {"window": "train (steps 1-504, days 1-21)", "transactions": len(X_train), "fraud_cases": int(y_train.sum()), "fraud_rate_pct": 100 * y_train.mean(), "total_value": float(amt_train.sum())},
    {"window": "validation (steps 505-624, days 22-26)", "transactions": len(X_val), "fraud_cases": int(y_val.sum()), "fraud_rate_pct": 100 * y_val.mean(), "total_value": float(amt_val.sum())},
    {"window": "test (steps 625-743, days 27-31)", "transactions": len(X_test), "fraud_cases": int(y_test.sum()), "fraud_rate_pct": 100 * y_test.mean(), "total_value": float(amt_test.sum())}
]).set_index("window")
split_table.to_csv(f"{OUT_DIR}/tables/M4_2_temporal_split_summary.csv")
display(split_table)

,transactions,fraud_cases,fraud_rate_pct,total_value
window,,,,
"train (steps 1-504, days 1-21)",2646363,5621,0.2124,"841,500,066,339.8899"
"validation (steps 505-624, days 22-26)",85954,1264,1.4706,"25,468,711,716.4500"
"test (steps 625-743, days 27-31)",38092,1328,3.4863,"12,736,204,431.3200"


In [4]:
# ===== [M4.3] Class Imbalance Handling & Feature Selection on Training Window =====
# 1. Class weight ratio for cost-sensitive classifiers (Logistic Regression, Random Forest, XGBoost, LightGBM)
n_pos_train = int((y_train == 1).sum())
n_neg_train = int((y_train == 0).sum())
scale_pos_weight = n_neg_train / max(n_pos_train, 1)
print(f"Training set class imbalance ratio (legitimate per fraud): {scale_pos_weight:,.2f}")

# 2. Random Undersampling (1:10 ratio) on training set for fast feature selection & resampling experiments
rng = np.random.default_rng(42)
pos_idx = np.where(y_train.to_numpy() == 1)[0]
neg_idx = np.where(y_train.to_numpy() == 0)[0]
sampled_neg_idx = rng.choice(neg_idx, size=min(len(neg_idx), len(pos_idx) * 10), replace=False)
rus_idx = np.sort(np.concatenate([pos_idx, sampled_neg_idx]))

X_train_rus = X_train.iloc[rus_idx].reset_index(drop=True)
y_train_rus = y_train.iloc[rus_idx].reset_index(drop=True)

# 3. Single-feature AUC, PR-AUC, and Random Forest Feature Importance on Training Window
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X_train_rus, y_train_rus)

feature_ranking = []
y_tr_np = y_train.to_numpy()
for i, col in enumerate(FEATURE_COLS):
    x_tr_np = X_train[col].to_numpy()
    auc_val = roc_auc_score(y_tr_np, x_tr_np)
    sep = abs(2 * auc_val - 1)
    pr_auc = average_precision_score(y_tr_np, x_tr_np if auc_val >= 0.5 else -x_tr_np)
    feature_ranking.append({
        "feature": col,
        "rf_importance": float(rf.feature_importances_[i]),
        "train_roc_auc": float(auc_val),
        "separation": float(sep),
        "train_pr_auc": float(pr_auc)
    })

feat_rank_df = pd.DataFrame(feature_ranking).set_index("feature").sort_values("rf_importance", ascending=False)
feat_rank_df.to_csv(f"{OUT_DIR}/tables/M4_3_feature_selection_ranking.csv")
display(feat_rank_df)

# Save engineered splits for Module 5
X_train.assign(isFraud=y_train.values, raw_amount=amt_train).to_parquet(f"{OUT_DIR}/train_fe_module4.parquet", index=False)
X_val.assign(isFraud=y_val.values, raw_amount=amt_val).to_parquet(f"{OUT_DIR}/val_fe_module4.parquet", index=False)
X_test.assign(isFraud=y_test.values, raw_amount=amt_test).to_parquet(f"{OUT_DIR}/test_fe_module4.parquet", index=False)
print("Saved engineered train/validation/test datasets for Module 5.")

Training set class imbalance ratio (legitimate per fraud): 469.80


,rf_importance,train_roc_auc,separation,train_pr_auc
feature,,,,
device_trust_score,0.2759,0.0062,0.9876,0.7581
failed_auth_attempts_24h,0.2347,0.9511,0.9022,0.4951
log_ip_billing_distance_km,0.1170,0.9486,0.8972,0.0931
ip_billing_distance_km,0.1082,0.9486,0.8972,0.0931
account_age_days,0.0868,0.1516,0.6968,0.4673
is_new_device,0.0612,0.8183,0.6367,0.0140
shipping_billing_mismatch,0.0401,0.7765,0.5530,0.0159
steps_since_last_dest_tx,0.0226,0.2937,0.4125,0.0061
prior_dest,0.0139,0.2350,0.5300,0.0064


Saved engineered train/validation/test datasets for Module 5.
